In [1]:

import json
import pennylane as qml
import pennylane.numpy as np

symbols = ["H", "H", "H"]

def h3_ground_energy(bond_length):
    """
    Uses VQE to calculate the ground energy of the H3+ molecule with the given bond length.

    Args:
        - bond_length(float): The bond length of the H3+ molecule modelled as an
          equilateral triangle.
    Returns:
        - Union[float, np.tensor, np.array]: A float-like output containing the ground 
          state of the H3+ molecule with the given bond length.
    """
    # Geometry for equilateral triangle in the xy-plane
    a = bond_length
    coords = np.array([
        [0.0, 0.0, 0.0],
        [a, 0.0, 0.0],
        [a / 2, a * np.sqrt(3) / 2, 0.0]
    ])

    charge = 1  # H3+ is +1 cation
    basis = "sto-3g"

    # Note: do NOT pass multiplicity as argument; use only charge and basis
    hamiltonian, qubits = qml.qchem.molecular_hamiltonian(
        symbols,
        coords,
        charge=charge,
        basis=basis
    )

    # n_electrons = sum(Z) - charge = 3 - 1 = 2
    n_electrons = sum(qml.qchem.atomic_number(s) for s in symbols) - charge

    # Hartree-Fock initial state for the given electron number
    hf_state = qml.qchem.hf_state(n_electrons, qubits)

    # All single and double excitations
    singles, doubles = qml.qchem.excitations(n_electrons, qubits)

    dev = qml.device("default.qubit", wires=qubits)

    # Define UCCSD ansatz as required
    def ansatz(params, wires):
        qml.BasisState(hf_state, wires=wires)
        qml.qchem.UCCSD(
            params,
            wires=wires,
            hf_state=hf_state,
            excitations=singles + doubles
        )

    n_params = len(singles) + len(doubles)

    @qml.qnode(dev)
    def circuit(params):
        ansatz(params, wires=range(qubits))
        return qml.expval(hamiltonian)

    # Initial params: small random values for optimizer stability
    np.random.seed(42)
    params = np.random.normal(0, 0.01, n_params)
    params = np.array(params, requires_grad=True)

    opt = qml.optimize.AdamOptimizer(stepsize=0.05)
    max_iters = 150
    conv_tol = 1e-6

    energy = [circuit(params)]
    for n in range(max_iters):
        params = opt.step(circuit, params)
        curr_energy = circuit(params)
        energy.append(curr_energy)
        if np.abs(energy[-1] - energy[-2]) < conv_tol:
            break

    return float(energy[-1])

def run(test_case_input: str) -> str:
    ins = json.loads(test_case_input)
    outs = h3_ground_energy(ins)
    return str(outs)

def check(solution_output: str, expected_output: str) -> None:
    solution_output = json.loads(solution_output)
    expected_output = json.loads(expected_output)
    assert np.allclose(solution_output, expected_output, atol=1e-4), "Not the correct ground energy"

# These are the public test cases
test_cases = [
    ('1.5', '-1.232574'),
    ('0.8', '-0.3770325')
]

# This will run the public test cases locally
for i, (input_, expected_output) in enumerate(test_cases):
    print(f"Running test case {i} with input '{input_}'...")

    try:
        output = run(input_)

    except Exception as exc:
        print(f"Runtime Error. {exc}")

    else:
        try:
            check(output, expected_output)
        except Exception as e:
            print(f"Wrong Answer. Have: '{output}'. Want: '{expected_output}'.")
        else:
            print("Correct!")


Running test case 0 with input '1.5'...
Runtime Error. _() got an unexpected keyword argument 'multiplicity'
Running test case 1 with input '0.8'...
Runtime Error. _() got an unexpected keyword argument 'multiplicity'
